# Paper II Preview: Stochastic Star Formation Histories

Scale from smooth 7-D to stochastic 137-D SFH inference using VI and MCMC samplers.

## What you'll learn

- **Gaussian process (GP) stochastic field** — add burstiness as traceable random field atop smooth mean
- **137-dimensional inference** — scale from 7 to 64-D GP + 7 physical parameters
- **Variational inference (VI)** — why geoVI dominates for high-D; seconds instead of hours
- **Ray Tracing Sampler** — exact MCMC tolerant of gradient noise for validation
- **SFR fluctuations at 10–100 Myr** — recover feedback-driven burstiness invisible to parametric models

## Prerequisites

[`00_quickstart.py`](00_quickstart.py) (smooth 7-D baseline). For PSD priors and
burstiness, see `examples/sfh/` gallery scripts (`plot_psd_*.py`).

**Paper II preview:** Advanced optional material; first-time users should finish Paper I spine first.

In [ ]:
import os
import sys
import time
import warnings

try:
    _nb_dir = os.path.dirname(os.path.abspath(__file__))
    _repo_root = os.path.abspath(os.path.join(_nb_dir, ".."))
except NameError:
    _nb_dir = os.getcwd()
    _repo_root = os.path.abspath(os.path.join(_nb_dir, ".."))

_src = os.path.join(_repo_root, "src")
if os.path.isdir(os.path.join(_src, "tengri")):
    sys.path.insert(0, _src)
sys.path.insert(0, _repo_root)
sys.path.insert(0, _nb_dir)

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore", category=FutureWarning)

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.45")

from tengri import (
    Fitter,
    Fixed,
    SEDModel,
    Observation,
    Parameters,
    Spectroscopy,
    Uniform,
    load_ssp_data,
)

# Locate ``notebooks/_plot_style.py`` and ``data/`` root (nbclient cwd is often wrong).
import importlib.util

_repo_data_root = None
_spec_tengri = importlib.util.find_spec("tengri")
if _spec_tengri is not None and _spec_tengri.origin:
    _walk = os.path.dirname(os.path.abspath(_spec_tengri.origin))
    for _step in range(12):
        _candidate = os.path.join(_walk, "notebooks", "_plot_style.py")
        if os.path.isfile(_candidate):
            sys.path.insert(0, os.path.dirname(_candidate))
            _repo_data_root = os.path.dirname(os.path.dirname(os.path.abspath(_candidate)))
            break
        _parent_walk = os.path.dirname(_walk)
        if _parent_walk == _walk:
            break
        _walk = _parent_walk

if _repo_data_root is None:
    _np_here = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(_np_here, "_plot_style.py")):
            sys.path.insert(0, _np_here)
            _repo_data_root = os.path.dirname(_np_here)
            break
        _ppt = os.path.join(_np_here, "notebooks", "_plot_style.py")
        if os.path.isfile(_ppt):
            _nbsd = os.path.dirname(_ppt)
            sys.path.insert(0, _nbsd)
            _repo_data_root = os.path.dirname(_nbsd)
            break
        _parent_here = os.path.dirname(_np_here)
        if _parent_here == _np_here:
            break
        _np_here = _parent_here

if _repo_data_root is not None and os.path.isdir(os.path.join(_repo_data_root, "data")):
    os.chdir(_repo_data_root)
elif os.path.isdir(os.path.join(_repo_root, "data")):
    os.chdir(_repo_root)
elif os.path.isdir("data"):
    pass
elif os.path.isdir(os.path.join("..", "data")):
    os.chdir("..")

FIGDIR = os.path.join("notebooks", "figures", "quickstart_stochastic")
os.makedirs(FIGDIR, exist_ok=True)

from _plot_style import (
    COLORS,
    SPECTRAL_FEATURES,
    convergence_table,
    plot_corner_comparison,
    plot_sfh,
    safe_corner,
    setup_style,
)

setup_style()

## Key Concepts (Quick Recap)

**Stochastic SFH** — The dense_basis or tsnorm parametric form gives the **mean** SFH.
Layered on top is a **Gaussian process (GP) field** ξ(t) that injects realistic
burstiness: rapid excursions away from the mean, autocorrelated on a physical timescale.

**Power Spectral Density (PSD)** — Controls the roughness of the GP. Two parameters:
- **σ_PS** (amplitude) — high = bursty, low = smooth
- **τ_PS** (timescale) — characteristic duration of bursts (e.g., 20 Myr = supernova feedback).

**Ray Tracing Sampler** — Exact MCMC sampler inspired by physics (light rays through parameter space,
Snell's law to bend toward high likelihood). ~250× more tolerant of gradient noise than NUTS;
works reliably at D = 137. See paper §4.

In [ ]:
# Memory / runtime gate. Default False runs the lightweight story (n_grid=64,
# variational only, no Ray Tracing). Flip to True to reproduce the paper §4
# figure (n_grid=128, variational + Ray Tracing exact MCMC on 137 dimensions).
RUN_EXPENSIVE = False

In [ ]:
# Load SSP templates; spectroscopy-only observation
WAVE_OBS = jnp.linspace(3800.0, 9200.0, 200)  # SDSS-like, 200 pixels
ssp_data = load_ssp_data("data/ssp_prsc_miles_chabrier_wNE_logGasU-3.0_logGasZ0.0.h5")
obs = Observation(spectroscopy=Spectroscopy(wave_obs=WAVE_OBS))
print(
    f"SSP templates: {ssp_data.ssp_flux.shape[0]} metallicities × {ssp_data.ssp_flux.shape[1]} ages "
    f"× {ssp_data.ssp_flux.shape[-1]} wavelengths"
)

## Bursty Galaxy — 137 Parameters

Real galaxies don't form stars smoothly. To recover star formation bursts (~1–100 Myr variability),
tengri uses a **GP field** with PSD priors. This adds 128 correlated latent dimensions to 9 physical
parameters: total free dimensionality = 137.

In [ ]:
# Define the stochastic parameter specification
# n_grid controls the discretisation of the GP field along lookback time.
# 128 is paper-figure resolution. With the stochastic model, Fitter.run() now
# auto-selects memory_mode="low" (jax.checkpoint on signal_response) so peak
# memory during CG stays bounded even at 137 dimensions on CPU.
spec_stoch = Parameters(
    sfh_tsnorm_log_peak_sfr=Uniform(-1.0, 2.5),
    sfh_tsnorm_peak_lbt_gyr=Uniform(0.5, 12.0),
    sfh_tsnorm_width_gyr=Uniform(0.3, 5.0),
    sfh_tsnorm_skew=Uniform(-3.0, 3.0),
    sfh_tsnorm_trunc=Uniform(1.0, 10.0),
    sfh_field_psd_sigma=Uniform(0.1, 4.0),
    sfh_field_psd_tau_myr=Uniform(1.0, 300.0),
    met_logzsol=Uniform(-2.0, 0.2),
    dust_tau_bc=Uniform(0.0, 2.0),
    dust_tau_diff=Uniform(0.0, 1.5),
    dust_slope=Fixed(-0.7),
    redshift=Fixed(0.1),
    mean_sfh_type=["tsnorm", "field"],
    n_grid=128,
)
print(f"Stochastic model: {spec_stoch.n_free} free parameters")
print(f"  Physical parameters: {len([p for p in spec_stoch.free_params if 'xi' not in p])}")
print(f"  GP latent dimensions (ξ): {len([p for p in spec_stoch.free_params if 'xi' in p])}")

In [ ]:
# Create stochastic model with spectroscopic precomputation
model_stoch = SEDModel(spec_stoch, ssp_data, observation=obs)
model_stoch.precompute_spectroscopy(WAVE_OBS)

In [ ]:
# Generate a bursty mock galaxy — feedback-driven regime with rapid SFR variability
key = jax.random.PRNGKey(123)
true_params_stoch = spec_stoch.sample(key)
# Override to a typical star-forming galaxy with dramatic burstiness
true_params_stoch = {**true_params_stoch}
true_params_stoch["sfh_tsnorm_log_peak_sfr"] = jnp.array(1.2)
true_params_stoch["sfh_tsnorm_peak_lbt_gyr"] = jnp.array(3.0)
true_params_stoch["sfh_tsnorm_width_gyr"] = jnp.array(3.0)
true_params_stoch["sfh_tsnorm_skew"] = jnp.array(0.3)
true_params_stoch["sfh_tsnorm_trunc"] = jnp.array(2.0)
true_params_stoch["sfh_field_psd_sigma"] = jnp.array(2.0)
true_params_stoch["sfh_field_psd_tau_myr"] = jnp.array(20.0)

mock_stoch = model_stoch.mock_spectrum(
    true_params_stoch, WAVE_OBS, snr=30.0, key=jax.random.fold_in(key, 1)
)
print("\nTrue PSD parameters (burstiness):")
print(f"  σ_PS (amplitude) = {float(true_params_stoch['sfh_field_psd_sigma']):.1f}")
print(f"  τ_PS (timescale) = {float(true_params_stoch['sfh_field_psd_tau_myr']):.0f} Myr")

In [ ]:
# --- FIGURE 1: The Bursty Truth (SFH + Spectrum) ---
sfh_true = model_stoch.predict_sfh(true_params_stoch)
t_gyr = np.array(sfh_true["t_gyr"])
sfr_full = np.array(sfh_true["sfr_full"])
sfr_mean = np.array(sfh_true["sfr_mean"])

fig, (ax_sfh, ax_spec) = plt.subplots(1, 2, figsize=(12, 4))

# Left: True SFH with burst structure
ax_sfh.plot(t_gyr, sfr_full, color=COLORS["truth"], lw=1.5, label="Full SFH (with GP bursts)")
ax_sfh.plot(t_gyr, sfr_mean, color=COLORS["sfh_mean"], lw=1, ls="--", label="Mean SFH (secular)")
ax_sfh.set_xlabel("Lookback time [Gyr]")
ax_sfh.set_ylabel(r"SFR [$M_\odot\,{\rm yr}^{-1}$]")
ax_sfh.set_xlim(0, 13.5)
ax_sfh.legend(fontsize=10)
ax_sfh.set_title("True Bursty SFH (σ_PS = 2.0, τ_PS = 20 Myr)")
# 200 Myr inset to highlight burst structure
inset = ax_sfh.inset_axes([0.55, 0.55, 0.4, 0.4])
mask_200 = t_gyr < 0.2
inset.plot(t_gyr[mask_200] * 1e3, sfr_full[mask_200], color=COLORS["truth"], lw=1)
inset.plot(t_gyr[mask_200] * 1e3, sfr_mean[mask_200], color=COLORS["sfh_mean"], lw=0.8, ls="--")
inset.set_xlabel("Lookback [Myr]", fontsize=10)
inset.set_ylabel("SFR", fontsize=10)
inset.tick_params(labelsize=5)
inset.set_xlim(0, 200)

# Right: Mock spectrum (SNR = 30)
ax_spec.errorbar(
    np.array(WAVE_OBS),
    np.array(mock_stoch.flux_obs),
    yerr=np.array(mock_stoch.noise),
    fmt=".",
    ms=2,
    color=COLORS["data"],
    alpha=0.5,
)
ax_spec.plot(np.array(WAVE_OBS), np.array(mock_stoch.flux_true), color=COLORS["truth"], lw=1)
ax_spec.set_xlabel("Observed wavelength [Å]")
ax_spec.set_ylabel("Flux density")
ax_spec.set_title("Mock Spectrum (SNR = 30)")

fig.tight_layout()
plt.show()

## Stochastic SFH Recovery: Fitting the Bursty Model

Now fit the 137-dimensional posterior. With standard MCMC this would be intractable.
But because the entire forward model is **differentiable**, variational inference
(using NIFTy's geometric variational inference) scales naturally.

In [ ]:
# vi (NIFTy geoVI) on the stochastic model — 137-D posterior
fitter_stoch = Fitter(
    model_stoch,
    mock_stoch.flux_obs,
    mock_stoch.noise,
)

t0 = time.perf_counter()
result_geovi_stoch = fitter_stoch.run(
    "vi",
    n_iterations=15,
    n_samples=3,
    n_posterior_samples=500,
    posterior_chunk_size=64,
    verbose=False,
)
t_geovi_s = time.perf_counter() - t0

print(f"\n{'=' * 60}")
print(f"  137-dimensional posterior in {t_geovi_s:.1f}s runtime (vi)")
print(f"{'=' * 60}")

In [ ]:
# --- FIGURE 2: Stochastic SFH Recovery (THE DISTINCTIVE FIGURE) ---
fig, ax = plt.subplots(figsize=(8, 4))
plot_sfh(
    model_stoch,
    result_geovi_stoch,
    true_params=true_params_stoch,
    ax=ax,
    color=COLORS["vi"],
    label="vi",
    method="geoVI",
    show_mean_sfh=True,
)
ax.set_title(
    f"Stochastic SFH Recovery — 137 parameters, {t_geovi_s:.1f}s runtime (vi)",
    fontweight="bold",
)
# 200 Myr inset — truth + posterior SFH draws showing burst recovery
sfh_true_s = model_stoch.predict_sfh(true_params_stoch)
t_gyr_s = np.array(sfh_true_s["t_gyr"])
sfr_key_s = "sfr_full"
sfr_true_s = np.array(sfh_true_s[sfr_key_s])
inset = ax.inset_axes([0.58, 0.58, 0.38, 0.38])
mask_200 = t_gyr_s < 0.2
if hasattr(t_gyr_s, "__len__") and np.any(mask_200):
    t_inset = t_gyr_s[mask_200] * 1e3
    if result_geovi_stoch.samples is not None:
        n_samp = len(next(iter(result_geovi_stoch.samples.values())))
        sfh_draws = []
        for i in range(n_samp):
            s_i = {k: result_geovi_stoch.samples[k][i] for k in result_geovi_stoch.samples}
            sfh_draws.append(np.array(model_stoch.predict_sfh(s_i)[sfr_key_s])[mask_200])
        sfh_arr = np.array(sfh_draws)
        lo, hi = np.percentile(sfh_arr, [16, 84], axis=0)
        median = np.median(sfh_arr, axis=0)
        inset.fill_between(t_inset, lo, hi, color=COLORS["vi"], alpha=0.3, lw=0)
        inset.plot(t_inset, median, color=COLORS["vi"], lw=1.2, label="Posterior median")
    inset.plot(t_inset, sfr_true_s[mask_200], color=COLORS["truth"], lw=1.5, label="Truth")
    inset.set_xlabel("Lookback [Myr]", fontsize=10)
    inset.set_ylabel("SFR", fontsize=10)
    inset.tick_params(labelsize=5)
    inset.set_xlim(0, 200)
    inset.legend(fontsize=10, loc="upper right")
fig.tight_layout()
plt.show()

In [ ]:
# --- FIGURE 3: Spectral Fit ---
spec_samples_s = []
for i in range(50):
    idx = i % len(result_geovi_stoch.samples[spec_stoch.free_params[0]])
    draw_params = {k: v[idx] for k, v in result_geovi_stoch.samples.items()}
    spec_draw = model_stoch.predict_spectrum(draw_params)
    spec_samples_s.append(np.array(spec_draw))
spec_samples_s = np.array(spec_samples_s)
spec_median_s = np.median(spec_samples_s, axis=0)

fig, (ax_fit, ax_res) = plt.subplots(
    2, 1, figsize=(10, 5), gridspec_kw={"height_ratios": [3, 1]}, sharex=True
)
wave_np = np.array(WAVE_OBS)
obs_s = np.array(mock_stoch.flux_obs)
noise_s = np.array(mock_stoch.noise)
true_s = np.array(mock_stoch.flux_true)

ax_fit.errorbar(wave_np, obs_s, yerr=noise_s, fmt=".", ms=2, color=COLORS["data"], alpha=0.4)
for s in spec_samples_s[:50]:
    ax_fit.plot(wave_np, s, color=COLORS["vi"], alpha=0.03, lw=0.5)
ax_fit.plot(wave_np, spec_median_s, color=COLORS["vi"], lw=1.5, label="vi (geoVI) median")
ax_fit.plot(wave_np, true_s, color=COLORS["truth"], lw=1, ls="--", label="Truth")
ax_fit.legend(fontsize=10)
ax_fit.set_ylabel("Flux density")

residuals_s = (obs_s - spec_median_s) / noise_s
ax_res.scatter(wave_np, residuals_s, s=2, c=COLORS["data"], alpha=0.5)
ax_res.axhline(0, color="k", lw=0.5)
ax_res.axhspan(-1, 1, alpha=0.1, color="grey")
ax_res.axhspan(-2, 2, alpha=0.05, color="grey")
ax_res.set_ylabel(r"$(f_{\rm obs} - f_{\rm model}) / \sigma$")
ax_res.set_xlabel("Observed wavelength [Å]")
ax_res.set_ylim(-4, 4)

chi2_s = np.sum(residuals_s**2) / len(residuals_s)
ax_fit.set_title(f"Spectral Fit — Stochastic (D = 137, reduced χ² = {chi2_s:.2f})")
fig.tight_layout()
plt.show()

In [ ]:
# Physical parameter corner (excluding the 128 latent ξ dimensions)
phys_params = [p for p in spec_stoch.free_params if "xi" not in p]
fig = safe_corner(result_geovi_stoch, truths=true_params_stoch, params=phys_params)
if fig is not None:
    fig.suptitle("Physical Parameters (D = 9) — Stochastic SFH Posterior", y=1.02)
plt.show()

## Exact MCMC Validation: Ray Tracing Sampler

To validate that vi (NIFTy geoVI) produces reliable posteriors at D = 137,
we use the **Ray Tracing Sampler** — an exact MCMC method that handles
high-dimensional posteriors without the pathology (premature mixing termination)
that plagues NUTS in high dimensions.

Set `RUN_EXPENSIVE = True` to run this 137-D chain (takes ~60s on CPU).

In [ ]:
# RUN_EXPENSIVE is declared at the top of the notebook — flip it there, not here.
if RUN_EXPENSIVE:
    t0 = time.perf_counter()
    result_rt_stoch = fitter_stoch.run(
        "mcmc_raytrace",
        init_from=result_geovi_stoch,
        n_burnin=200,
        n_steps=2000,
        step_size=0.05,
        n_leapfrog_steps=50,
        verbose=False,
    )
    t_rt_s = time.perf_counter() - t0
    acc = result_rt_stoch.diagnostics.get("acceptance_rate", float("nan"))
    print(f"Ray Tracing: {t_rt_s:.1f}s, acceptance = {acc:.1%}")

    # Convergence diagnostics for D=137 methods
    ct_stoch = convergence_table({"vi": result_geovi_stoch, "Ray Tracing": result_rt_stoch})

    # --- FIGURE 4: Method Comparison (vi vs Ray Tracing) ---
    fig, ax = plt.subplots(figsize=(10, 5))

    # Truth first — bold, on top
    sfh_true_cmp = model_stoch.predict_sfh(true_params_stoch)
    t_gyr_cmp = np.array(sfh_true_cmp["t_gyr"])
    ax.plot(
        t_gyr_cmp,
        sfh_true_cmp["sfr_full"],
        color=COLORS["truth"],
        lw=3.0,
        zorder=10,
        label="Truth",
    )
    ax.plot(
        t_gyr_cmp,
        sfh_true_cmp["sfr_mean"],
        color=COLORS["truth"],
        lw=1.5,
        ls=":",
        alpha=0.4,
        zorder=10,
    )

    # Overlay each method's posterior SFH as 68% CI band
    for result, color, label in [
        (result_geovi_stoch, COLORS["vi"], f"vi (NIFTy) ({t_geovi_s:.1f}s)"),
        (result_rt_stoch, COLORS["rt"], f"Ray Tracing ({t_rt_s:.1f}s)"),
    ]:
        if result.samples is not None:
            n_samp = len(next(iter(result.samples.values())))
            sfh_draws = []
            for i in range(n_samp):
                s_i = {k: result.samples[k][i] for k in result.samples}
                sfh_draws.append(np.array(model_stoch.predict_sfh(s_i)["sfr_full"]))
            sfh_arr = np.array(sfh_draws)
            lo, hi = np.percentile(sfh_arr, [16, 84], axis=0)
            median = np.median(sfh_arr, axis=0)
            ax.fill_between(
                t_gyr_cmp, lo, hi, color=color, alpha=0.2, lw=0, label=f"{label} (68% CI)"
            )
            ax.plot(t_gyr_cmp, median, color=color, lw=1.5, zorder=5)

    ax.set_xlabel(r"$\mathrm{Lookback\ time\ /\ Gyr}$")
    ax.set_ylabel(r"$\mathrm{SFR\ /\ M_\odot\ yr^{-1}}$")
    ax.set_xlim(0, 13.5)
    ax.set_ylim(bottom=0.0)
    ax.legend(loc="upper right")
    ax.set_title("Stochastic SFH Recovery — vi (NIFTy) vs Ray Tracing (D = 137)")

    # 200 Myr inset with both posteriors
    inset = ax.inset_axes([0.55, 0.55, 0.4, 0.4])
    mask_cmp = t_gyr_cmp < 0.2
    if np.any(mask_cmp):
        t_inset = t_gyr_cmp[mask_cmp] * 1e3
        for result, color, label in [
            (result_geovi_stoch, COLORS["vi"], "NIFTy"),
            (result_rt_stoch, COLORS["rt"], "RT"),
        ]:
            if result.samples is not None:
                n_samp = len(next(iter(result.samples.values())))
                draws = []
                for i in range(n_samp):
                    s_i = {k: result.samples[k][i] for k in result.samples}
                    draws.append(np.array(model_stoch.predict_sfh(s_i)["sfr_full"])[mask_cmp])
                arr = np.array(draws)
                lo, hi = np.percentile(arr, [16, 84], axis=0)
                med = np.median(arr, axis=0)
                inset.fill_between(t_inset, lo, hi, color=color, alpha=0.25, lw=0)
                inset.plot(t_inset, med, color=color, lw=1.2, label=label)
        inset.plot(
            t_inset,
            np.array(sfh_true_cmp["sfr_full"])[mask_cmp],
            color=COLORS["truth"],
            lw=2.0,
            label="Truth",
        )
        inset.set_xlabel("Lookback [Myr]", fontsize=10)
        inset.set_ylabel("SFR", fontsize=10)
        inset.tick_params(labelsize=5)
        inset.set_xlim(0, 200)
        inset.legend(fontsize=10, loc="upper right")

    fig.tight_layout()
    plt.show()
else:
    print("\nTo reproduce paper §4 figure (exact MCMC on 137D posterior):")
    print("  Set RUN_EXPENSIVE = True")
    print("  Ray Tracing with 2000 steps + burnin takes ~60s on CPU")

## Summary & Takeaways

In [ ]:
# Summary timing table
print("\n" + "=" * 70)
print(f"  {'Model':<18s} {'D':>5s} {'Method':<16s} {'Runtime':>8s} Notes")
print("=" * 70)
print(
    f"  {'Stochastic':<18s} {'137':>5s} {'vi (NIFTy)':<16s} {t_geovi_s:>7.1f}s   Default (paper §4)"
)
if RUN_EXPENSIVE:
    print(
        f"  {'Stochastic':<18s} {'137':>5s} {'Ray Tracing':<16s} {t_rt_s:>7.1f}s   Exact (Behroozi 2025)"
    )
print("=" * 70)
print(f"\nHeadline: 137D posterior in ~{t_geovi_s:.0f}s via variational inference (vi).")
print("Validation via exact MCMC (Ray Tracing) confirms reliability.")

## What You Just Did

1. **Activated the stochastic field**: 128 correlated latent dimensions capturing realistic burstiness.
2. **Fitted the 137D posterior** in seconds using vi (geoVI) — impossible for NUTS.
3. **Recovered ~10–100 Myr burst features** from a single optical spectrum.
4. **Validated against exact MCMC** (Ray Tracing Sampler) to confirm posterior reliability.

## What's Next

- **Smooth SFH baseline:** [`00_quickstart.py`](00_quickstart.py) — the 7-parameter story you built from.
- **PSD theory and priors:** `examples/sfh/plot_psd_*.py` — mathematical foundations.
- **Extending tengri:** [`13_extending_tengri.py`](13_extending_tengri.py) — custom inference and SFH forms.
- **Full gallery spine:** `03` (dust) → `04` (nebular) → `05` (AGN) → ... → `15` (emission lines).

The stochastic model is Paper II's centerpiece. You now understand how differentiable physics
enables exact inference in dimensions where traditional samplers fail.

## What you learned

- Stochastic SFH adds 64-D GP field encoding burst morphology (timescale tau_PS, amplitude sigma_PS)
- VI scaling: 137-D fits in seconds; NUTS would run for hours
- Ray Tracing Sampler validates VI posteriors without full sampling cost
- Individual galaxies cannot constrain tau_PS; population hierarchical priors solve this (see 11_population.py)

**Next:** [`15_vi_inference.py`](15_vi_inference.py) (VI scaling theory and benchmarks) or
[`16_simulation_interface.py`](16_simulation_interface.py) (forward-modeling simulation outputs).